In [12]:
import re
import pdfplumber
import pandas as pd
from collections import defaultdict

PDF_PATH = "2023_Dealer_List.pdf"

# -----------------------------
# 1. Basic text cleaning
# -----------------------------
def clean_text(text: str) -> str:
    if text is None:
        return ""
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


# -----------------------------
# 2. Group words into rows
# -----------------------------
def group_words_into_rows(words, y_tolerance=3):
    rows = defaultdict(list)

    for w in words:
        row_key = round(w["top"] / y_tolerance) * y_tolerance
        rows[row_key].append(w)

    grouped = []
    for row_key in sorted(rows.keys()):
        row_words = sorted(rows[row_key], key=lambda x: x["x0"])
        grouped.append((row_key, row_words))

    return grouped


# -----------------------------
# 3. Fixed column assignment
# -----------------------------
# These boundaries are chosen from the actual page layout:
# Name      starts around x=23
# Address   starts around x=203
# City      starts around x=373
# State     starts around x=498
# Zip Code  starts around x=534
#
# We therefore use stable midpoints between the printed headers.
def assign_column(x_center):
    if x_center < 190:
        return "dealer_name"
    elif x_center < 360:
        return "address"
    elif x_center < 490:
        return "city"
    elif x_center < 530:
        return "state"
    else:
        return "zip_code"


# -----------------------------
# 4. Post-process address/city boundary issues
# -----------------------------
DIRECTION_TOKENS = {
    "N", "S", "E", "W",
    "NE", "NW", "SE", "SW",
    "North", "South", "East", "West",
    "N.", "S.", "E.", "W.",
    "N.E.", "N.W.", "S.E.", "S.W."
}


def fix_address_city_boundary(address: str, city: str) -> tuple[str, str]:
    """Move stray direction tokens from city back to address.

    Examples:
    - 'N Albertville'   -> address + 'N', city='Albertville'
    - 'East Anniston'   -> address + 'East', city='Anniston'
    - 'N.W. Huntsville' -> address + 'N.W.', city='Huntsville'
    """
    address = clean_text(address)
    city = clean_text(city)

    if not city:
        return address, city

    parts = city.split()
    if len(parts) >= 2 and parts[0] in DIRECTION_TOKENS:
        address = clean_text(f"{address} {parts[0]}")
        city = clean_text(" ".join(parts[1:]))

    return address, city


# -----------------------------
# 5. Extract dealer table
# -----------------------------
def extract_dealer_table(pdf_path: str) -> pd.DataFrame:
    all_rows = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            words = page.extract_words(keep_blank_chars=False, use_text_flow=False)
            grouped_rows = group_words_into_rows(words, y_tolerance=3)

            for row_top, row_words in grouped_rows:
                row_text = " ".join(clean_text(w["text"]) for w in row_words).strip()
                lower_row_text = row_text.lower()

                # Skip title/header rows
                if not row_text:
                    continue
                if "toyota dealers as of" in lower_row_text:
                    continue
                if lower_row_text in {"name address city state zip code", "name address city state zip code code"}:
                    continue

                row_dict = {
                    "dealer_name": [],
                    "address": [],
                    "city": [],
                    "state": [],
                    "zip_code": [],
                    "page": page_num,
                }

                for w in row_words:
                    text = clean_text(w["text"])
                    x_center = (w["x0"] + w["x1"]) / 2
                    col = assign_column(x_center)
                    row_dict[col].append(text)

                parsed = {
                    "dealer_name": clean_text(" ".join(row_dict["dealer_name"])),
                    "address": clean_text(" ".join(row_dict["address"])),
                    "city": clean_text(" ".join(row_dict["city"])),
                    "state": clean_text(" ".join(row_dict["state"])),
                    "zip_code": clean_text(" ".join(row_dict["zip_code"])),
                    "page": row_dict["page"],
                }

                # Basic row filter
                if not parsed["dealer_name"]:
                    continue
                if not re.fullmatch(r"[A-Z]{2}", parsed["state"]):
                    continue
                if not re.search(r"\d{5}", parsed["zip_code"]):
                    continue

                # Fix boundary issues between address and city
                parsed["address"], parsed["city"] = fix_address_city_boundary(
                    parsed["address"], parsed["city"]
                )

                # Keep only first 5-digit ZIP
                zip_match = re.search(r"\d{5}", parsed["zip_code"])
                parsed["zip_code"] = zip_match.group(0) if zip_match else ""

                all_rows.append(parsed)

    return pd.DataFrame(all_rows)


# -----------------------------
# 6. Run extraction
# -----------------------------
df_dealers = extract_dealer_table(PDF_PATH)

print(df_dealers.head(25))
print(df_dealers.shape)

# Save main parsed file
df_dealers.to_csv("toyota_dealers_parsed_better.csv", index=False)

# -----------------------------
# 7. Aggregations
# -----------------------------
dealer_count_by_state = (
    df_dealers.groupby("state", as_index=False)
    .size()
    .rename(columns={"size": "dealer_count"})
    .sort_values(["dealer_count", "state"], ascending=[False, True])
)

dealer_count_by_state_city = (
    df_dealers.groupby(["state", "city"], as_index=False)
    .size()
    .rename(columns={"size": "dealer_count"})
    .sort_values(["state", "dealer_count", "city"], ascending=[True, False, True])
)

dealer_count_by_state.to_csv("dealer_count_by_state_better.csv", index=False)
dealer_count_by_state_city.to_csv("dealer_count_by_state_city_better.csv", index=False)

# Optional Excel output
with pd.ExcelWriter("toyota_dealer_outputs_better.xlsx", engine="openpyxl") as writer:
    df_dealers.to_excel(writer, sheet_name="parsed_dealers", index=False)
    dealer_count_by_state.to_excel(writer, sheet_name="count_by_state", index=False)
    dealer_count_by_state_city.to_excel(writer, sheet_name="count_by_state_city", index=False)

print("Saved:")
print("- toyota_dealers_parsed_better.csv")
print("- dealer_count_by_state_better.csv")
print("- dealer_count_by_state_city_better.csv")
print("- toyota_dealer_outputs_better.xlsx")


                      dealer_name                     address          city  \
0     Kendall Toyota of Anchorage     6930 Old Seward Highway     Anchorage   
1     Kendall Toyota of Fairbanks         1000 Cadillac Court     Fairbanks   
2                   Juneau Toyota            8602 Teal Street        Juneau   
3            Sand Mountain Toyota       9167 US Highway 431 N   Albertville   
4               Sunny King Toyota   2570 U.S. Highway 78 East      Anniston   
5          Lynch Toyota of Auburn      170 West Creek Parkway        Auburn   
6                 Limbaugh Toyota               2200 Avenue T    Birmingham   
7                    Serra Toyota   1300 Center Point Parkway    Birmingham   
8                 McKinnon Toyota             235 Price Drive       Clanton   
9            Eastern Shore Toyota       29732 Frederick Blvd.        Daphne   
10        Serra Toyota of Decatur          309 Beltline Place       Decatur   
11               Toyota of Dothan      2285 Ross Cla

In [13]:
df = pd.read_csv("toyota_dealers_parsed_better.csv")

df.loc[
    df["dealer_name"].eq("SVG Toyota") & df["state"].eq("OH") & df["zip_code"].astype(str).eq("43160"),
    "city"
] = "Washington Court House"

df.to_csv("toyota_dealers_parsed_checked.csv", index=False)


In [14]:
import pandas as pd

# Load the cleaned dealer-level dataset
df = pd.read_csv("toyota_dealers_parsed_checked.csv")

# Aggregate dealer counts by city and state
city_state_counts = (
    df.groupby(["city", "state"], as_index=False)
      .size()
      .rename(columns={
          "city": "city_name",
          "state": "state",
          "size": "dealer_count"
      })
      .sort_values(
          ["state", "dealer_count", "city_name"],
          ascending=[True, False, True]
      )
)

# Export results to CSV
city_state_counts.to_csv("city_state_dealer_counts.csv", index=False)

# Export results to Excel
city_state_counts.to_excel("city_state_dealer_counts.xlsx", index=False)

# Preview top rows
print(city_state_counts.head(20))

print("Files saved:")
print("- city_state_dealer_counts.csv")
print("- city_state_dealer_counts.xlsx")

         city_name state  dealer_count
24       Anchorage    AK             1
337      Fairbanks    AK             1
530         Juneau    AK             1
95      Birmingham    AL             2
704         Mobile    AL             2
9      Albertville    AL             1
29        Anniston    AL             1
45          Auburn    AL             1
197        Clanton    AL             1
257         Daphne    AL             1
266        Decatur    AL             1
283         Dothan    AL             1
327     Enterprise    AL             1
480         Hoover    AL             1
493     Huntsville    AL             1
519         Jasper    AL             1
710     Montgomery    AL             1
863   Rainbow City    AL             1
949     Scottsboro    AL             1
1021     Sylacauga    AL             1
Files saved:
- city_state_dealer_counts.csv
- city_state_dealer_counts.xlsx


In [15]:
import pandas as pd
import numpy as np
import re

# ============================================
# 1. Read the two input files
# ============================================
uscities = pd.read_csv("uscities.csv")
dealers = pd.read_csv("city_state_dealer_counts.csv")

# ============================================
# 2. Rename key columns
# ============================================
uscities = uscities.rename(columns={
    "city": "city_name",
    "state_name": "state_full",
    "state_id": "state_abbr"
})

dealers = dealers.rename(columns={
    "state": "dealer_state_abbr"
})

# ============================================
# 3. Normalise city names for matching
# ============================================
def normalize_text(x):
    if pd.isna(x):
        return ""
    x = str(x).strip().lower()
    x = re.sub(r"\s+", " ", x)
    x = x.replace(".", "")
    x = x.replace("-", " ")
    x = x.replace("'", "")
    return x

uscities["city_key"] = uscities["city_name"].apply(normalize_text)
dealers["city_key"] = dealers["city_name"].apply(normalize_text)

uscities["state_abbr"] = uscities["state_abbr"].astype(str).str.strip().str.upper()
dealers["dealer_state_abbr"] = dealers["dealer_state_abbr"].astype(str).str.strip().str.upper()

# ============================================
# 4. Prepare dealer tables
# ============================================
# A. Full dealer table for exact match: city + state
dealers_exact = dealers[["city_key", "dealer_state_abbr", "dealer_count"]].copy()

# B. City-only table to check whether city name exists at all
#    If one city name appears in multiple states, keep unique city_key only.
dealers_city_only = dealers[["city_key"]].drop_duplicates().copy()
dealers_city_only["city_exists_in_dealer_file"] = 1

# ============================================
# 5. Merge exact dealer_count by city + state
# ============================================
merged = uscities.merge(
    dealers_exact,
    left_on=["city_key", "state_abbr"],
    right_on=["city_key", "dealer_state_abbr"],
    how="left"
)

# Fill unmatched dealer_count with 0
merged["dealer_count"] = merged["dealer_count"].fillna(0).astype(int)

# ============================================
# 6. Add state_match column
# ============================================
# First check whether city name appears anywhere in dealer file
merged = merged.merge(
    dealers_city_only,
    on="city_key",
    how="left"
)

merged["city_exists_in_dealer_file"] = merged["city_exists_in_dealer_file"].fillna(0).astype(int)

# state_match:
# 1 -> city exists in dealer file AND exact city+state match succeeded
# 0 -> otherwise
merged["state_match"] = np.where(
    (merged["city_exists_in_dealer_file"] == 1) & (merged["dealer_count"] > 0),
    1,
    0
)

# ============================================
# 7. Convert military / incorporated to 0/1
# ============================================
def convert_bool_like_to_int(series):
    # Handle True/False strings and actual booleans
    s = series.copy()

    # If already boolean
    if pd.api.types.is_bool_dtype(s):
        return s.astype(int)

    # Otherwise convert from string-like values
    s = (
        s.astype(str)
         .str.strip()
         .str.lower()
         .map({
             "true": 1,
             "false": 0,
             "1": 1,
             "0": 0
         })
    )
    return s

if "military" in merged.columns:
    merged["military"] = convert_bool_like_to_int(merged["military"])

if "incorporated" in merged.columns:
    merged["incorporated"] = convert_bool_like_to_int(merged["incorporated"])

# ============================================
# 8. Clean final output columns
# ============================================
# Remove helper columns you do not need in final table
drop_cols = [
    "city_key",
    "dealer_state_abbr",
    "city_exists_in_dealer_file"
]

existing_drop_cols = [c for c in drop_cols if c in merged.columns]
merged = merged.drop(columns=existing_drop_cols)

# Put required columns at the front
front_cols = ["city_name", "state_full", "state_abbr", "dealer_count"]
other_cols = [c for c in merged.columns if c not in front_cols + ["state_match"]]

merged_final = merged[front_cols + other_cols + ["state_match"]].copy()

# ============================================
# 9. Save outputs
# ============================================
merged_final.to_csv("merged_city_dealers_all_columns_exact.csv", index=False)

with pd.ExcelWriter("merged_city_dealers_all_columns_exact.xlsx", engine="openpyxl") as writer:
    merged_final.to_excel(writer, sheet_name="merged_city_dealers", index=False)

# ============================================
# 10. Preview
# ============================================
print(merged_final.head(20))
print("\nSaved:")
print("- merged_city_dealers_all_columns_exact.csv")
print("- merged_city_dealers_all_columns_exact.xlsx")
print("\nShape:", merged_final.shape)

        city_name            state_full state_abbr  dealer_count  \
0        New York              New York         NY             1   
1     Los Angeles            California         CA             2   
2         Chicago              Illinois         IL             4   
3           Miami               Florida         FL             3   
4         Houston                 Texas         TX             6   
5          Dallas                 Texas         TX             4   
6    Philadelphia          Pennsylvania         PA             1   
7         Atlanta               Georgia         GA             2   
8      Washington  District of Columbia         DC             0   
9          Boston         Massachusetts         MA             2   
10        Phoenix               Arizona         AZ             2   
11        Detroit              Michigan         MI             0   
12        Seattle            Washington         WA             2   
13  San Francisco            California         

In [16]:
import pandas as pd
import re

# =========================
# 1. Load the two input datasets
# =========================
dealers = pd.read_csv("city_state_dealer_counts.csv")
uscities = pd.read_csv("uscities.csv")

# =========================
# 2. Standardize column names
# =========================
dealers = dealers.rename(columns={
    "state": "state_abbr"
})

uscities_sub = uscities.rename(columns={
    "city": "city_name",
    "state_id": "state_abbr",
    "state_name": "state_full"
})

# =========================
# 3. Define text normalization function
# =========================
def normalize_text(x):
    if pd.isna(x):
        return ""
    x = str(x).strip().lower()
    x = re.sub(r"\s+", " ", x)
    x = x.replace(".", "")
    x = x.replace("-", " ")
    x = x.replace("'", "")
    return x

# Normalize key matching fields
dealers["city_key"] = dealers["city_name"].apply(normalize_text)
dealers["state_key"] = dealers["state_abbr"].apply(normalize_text)

uscities_sub["city_key"] = uscities_sub["city_name"].apply(normalize_text)
uscities_sub["state_key"] = uscities_sub["state_abbr"].apply(normalize_text)

# =========================
# 4. Extract unique city-state combinations for validation
# =========================
dealer_keys = dealers[["city_name", "state_abbr", "dealer_count", "city_key", "state_key"]].drop_duplicates()
uscity_keys = uscities_sub[["city_name", "state_abbr", "city_key", "state_key"]].drop_duplicates()

# =========================
# 5. Anti-join: identify city-state pairs present in dealer data but missing from uscities
# =========================
check_merge = dealer_keys.merge(
    uscity_keys[["city_key", "state_key"]],
    on=["city_key", "state_key"],
    how="left",
    indicator=True
)

unmatched = check_merge[check_merge["_merge"] == "left_only"].copy()

print("Total dealer city-state pairs:", len(dealer_keys))
print("Successfully matched pairs:", (check_merge["_merge"] == "both").sum())
print("Unmatched pairs:", len(unmatched))

print("\nUnmatched city-state combinations:")
print(
    unmatched[["city_name", "state_abbr", "dealer_count"]]
    .sort_values(["state_abbr", "city_name"])
)

# Export unmatched results
unmatched[["city_name", "state_abbr", "dealer_count"]].to_csv(
    "unmatched_dealer_cities.csv",
    index=False
)

print("\nSaved: unmatched_dealer_cities.csv")

Total dealer city-state pairs: 1154
Successfully matched pairs: 1036
Unmatched pairs: 118

Unmatched city-state combinations:
             city_name state_abbr  dealer_count
73    City Of Industry         CA             1
103          Hollywood         CA             1
108       La Crescenta         CA             1
109         Lake Tahoe         CA             1
125         Northridge         CA             1
...                ...        ...           ...
1079            Berlin         VT             1
1083       Westminster         VT             1
1116             Allis         WI             1
1119              Bend         WI             1
1144      Saint Albans         WV             1

[118 rows x 3 columns]

Saved: unmatched_dealer_cities.csv
